In [1]:
import re
import pandas as pd
from typing import  Literal,List,Any
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.types import Command
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from langchain_core.prompts.chat import ChatPromptTemplate
from langgraph.graph import START, StateGraph,END
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field, field_validator
from langchain_core.messages import HumanMessage,AIMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv(dotenv_path="E:\Doctor-Appointment-Aiagent\.env")
import os
OPENAI_API_KEY=os.getenv("GROQ_API_KEY")

In [3]:
# openai_model=ChatOpenAI(model='gpt-4o')

In [16]:
data=pd.read_csv('E:\Doctor-Appointment-Aiagent\data\doctor.csv')

In [17]:
data.head()

,date_slot,specialization,doctor_name,is_available,patient_to_attend
0,05-08-2024 08:00,general_dentist,john doe,True,NaN
1,05-08-2024 08:30,general_dentist,john doe,False,1000082.0
2,05-08-2024 09:00,general_dentist,john doe,False,1000048.0
3,05-08-2024 09:30,general_dentist,john doe,False,1000036.0
4,05-08-2024 10:00,general_dentist,john doe,False,1000024.0


In [3]:
groq_model=ChatGroq(model="deepseek-r1-distill-llama-70b",api_key=OPENAI_API_KEY)

In [4]:
groq_model.invoke("hi")

AIMessage(content='<think>\n\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 4, 'total_tokens': 20, 'completion_time': 0.089238607, 'prompt_time': 9.8428e-05, 'queue_time': 0.21000502799999998, 'total_time': 0.089337035}, 'model_name': 'deepseek-r1-distill-llama-70b', 'system_fingerprint': 'fp_1bbe7845ec', 'finish_reason': 'stop', 'logprobs': None}, id='run-e21feab6-34b3-4baa-98c1-61e10183a788-0', usage_metadata={'input_tokens': 4, 'output_tokens': 16, 'total_tokens': 20})

In [5]:
# Import BaseModel and Field from Pydantic for data validation, and also import field_validator for custom validation
from pydantic import BaseModel, Field, field_validator
import re  # Import regular expressions module for pattern matching

# Define a Pydantic model named DateTimeModel
class DateTimeModel(BaseModel):
    # Define a field 'date' of type string with a description and a regex pattern to validate the format 'DD-MM-YYYY HH:MM'
    date: str = Field(
        description='properly formatted date', 
        pattern=r'^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    )

    # Define a validator for the 'date' field
    @field_validator('date')
    def check_date_formate(cls, v):
        # Check if the provided date string matches the required pattern
        if not re.match(r'^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$', v):
            # If not, raise a validation error with a clear message
            raise ValueError("The date should be in format 'DD-MM-YYYY HH:MM'")
        # If the format is correct, return the value
        return v


In [6]:
# Import necessary components from Pydantic
from pydantic import BaseModel, Field, field_validator
import re  # Import regular expressions module to validate string patterns

# Define a Pydantic model named DateModel
class DateModel(BaseModel):
    # Define a field 'date' which is a string
    # Add a description and enforce a regex pattern to match 'DD-MM-YYYY' format
    date: str = Field(
        description="Properly formatted date", 
        pattern=r'^\d{2}-\d{2}-\d{4}$'  # Regex ensures the date format is exactly DD-MM-YYYY
    )

    # Define a custom validator method for the 'date' field
    @field_validator('date')
    def check_formate_date(cls, v):
        # Use regex to check if the input matches the 'DD-MM-YYYY' pattern
        if not re.match(r'^\d{2}-\d{2}-\d{4}$', v):  # Ensures DD-MM-YYYY format strictly
            # Raise a ValueError with a clear message if format is incorrect
            raise ValueError("The date must be in the format 'DD-MM-YYYY'")
        # If validation passes, return the input value
        return v

        
        

In [7]:
# Import necessary components from Pydantic
from pydantic import BaseModel, Field, field_validator
import re  # Import regular expressions module for pattern matching

# Define a Pydantic model named IdentificationNumberModel
class IdentificationNumberModel(BaseModel):
    # Define a field 'id' which is an integer
    # Add a description specifying it must be 7 or 8 digits long
    id: int = Field(
        description='Identification number (7 or 8 digits long)'
    )

    # Define a custom validator method for the 'id' field
    @field_validator('id')
    def check_formate_id(cls, v):
        # Convert the integer 'v' to a string to apply regex pattern matching
        if not re.match(r'^\d{7,8}$', str(v)):  # Ensures that the ID is exactly 7 or 8 digits
            # Raise a ValueError with a clear message if the format is wrong
            raise ValueError("The ID number should be a 7 or 8-digit number")
        # If validation passes, return the original integer value
        return v

        

In [14]:
# Import necessary modules
import pandas as pd
from typing import Literal
# Assume DateModel is already imported

# Define a function to check the availability of a specific doctor on a specific date
@tool
def check_availability_by_doctor(
    desired_date: DateModel,  # The desired date passed as a DateModel instance
    doctor_name: Literal[
        'kevin anderson', 'robert martinez', 'susan davis', 'daniel miller',
        'sarah wilson', 'michael green', 'lisa brown', 'jane smith',
        'emily johnson', 'john doe'  # Restricts doctor names to a fixed list of valid options
    ]
):
    """
    Checks the database if a specific doctor has available slots on a given date.
    Parameters are provided by the user in the query.
    """

    # Load the doctor appointment dataset from a CSV file
    df = pd.read_csv('E:\Doctor-Appointment-Aiagent\data\doctor.csv')
    print(df['date_slot'])

    # Create a new column 'date_slot_time' by extracting the time part from 'date_slot'
    df['date_slot_time'] = df['date_slot'].apply(lambda input: input.split(' ')[-1])
    print(df['date_slot_time'])

    # Filter the dataframe:
    # - Match the date part of 'date_slot' with desired_date.date
    # - Match the 'doctor_name' exactly
    # - Only consider rows where 'is_available' is True
    rows = list(
        df[
            (df['date_slot'].apply(lambda input: input.split(' ')[0]) == desired_date.date) &  # Match date
            (df['doctor_name'] == doctor_name) &  # Match doctor name
            (df['is_available'] == True)  # Doctor must be available
        ]['date_slot_time']  # Select only the time slots
    )

    # If no available slots found
    if len(rows) == 0:
        output = "No availability in the entire day"
    else:
        # If slots found, prepare the output string
        output = f'This availability for {desired_date.date}\n'
        output += "Available slots: " + ', '.join(rows)

    # Return the final output
    return output


In [10]:
date_instance=DateModel(date="03-09-2024")

In [11]:
date_instance

DateModel(date='03-09-2024')

In [15]:

print(check_availability_by_doctor.invoke({"desired_date": date_instance, "doctor_name": "kevin anderson"}))

0       05-08-2024 08:00
1       05-08-2024 08:30
2       05-08-2024 09:00
3       05-08-2024 09:30
4       05-08-2024 10:00
              ...       
4275    03-09-2024 14:30
4276    03-09-2024 15:00
4277    03-09-2024 15:30
4278    03-09-2024 16:00
4279    03-09-2024 16:30
Name: date_slot, Length: 4280, dtype: object
0       08:00
1       08:30
2       09:00
3       09:30
4       10:00
        ...  
4275    14:30
4276    15:00
4277    15:30
4278    16:00
4279    16:30
Name: date_slot_time, Length: 4280, dtype: object
This availability for 03-09-2024
Available slots: 08:00, 08:30, 11:30, 12:00, 12:30, 13:00, 14:00, 14:30, 15:00, 15:30, 16:00, 16:30


In [ ]:
# Import necessary modules
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from langchain.tools import tool  # Assuming you are using LangChain's @tool decorator

# Define a function to check doctor availability by specialization
@tool
def check_availability_by_specialization(
    desired_date: DateModel,  # The desired date passed as a DateModel instance
    specialization: Literal[
        "general_dentist", "cosmetic_dentist", "prosthodontist",
        "pediatric_dentist", "emergency_dentist", "oral_surgeon", "orthodontist"  # Allowed specializations
    ]
):
    """
    Checks the database if we have availability for a specific specialization.
    The parameters should be mentioned by the user in the query.
    """

    # Read the doctor appo df = pd.read_csv('E:\Doctor-Appointment-Aiagent\data\doctor.csv')  # You forgot to add the CSV path hereintment dataset from a CSV file
   
    df = pd.read_csv('E:\Doctor-Appointment-Aiagent\data\doctor.csv')
    # Create a new column 'date_slot_time' by extracting the time from 'date_slot'
    df['date_slot_time'] = df['date_slot'].apply(lambda input: input.split(' ')[-1])

    # Filter the dataframe:
    # - Match the date from 'date_slot' with desired_date.date
    # - Match the specialization exactly
    # - Ensure that the slot is available (is_available == True)
    # Then group by 'specialization' and 'doctor_name' and collect available time slots into a list
    rows = df[
        (df['date_slot'].apply(lambda input: input.split(' ')[0]) == desired_date.date) &
        (df['specialization'] == specialization) &
        (df['is_available'] == True)
    ].groupby(['specialization', 'doctor_name'])['date_slot_time'].apply(list).reset_index(name='available_slots')

    # If no available slots are found
    if len(rows) == 0:
        output = "No availability in the entire day"
    else:
        # Define a helper function to convert 24-hour time format to 12-hour AM/PM format
        def convert_to_am_pm(time_str):
            time_str = str(time_str)  # Make sure the time is treated as a string
            hours, minutes = map(int, time_str.split(':'))  # Split into hours and minutes
            period = "AM" if hours < 12 else "PM"  # Determine if it is AM or PM
            hours = hours % 12 or 12  # Convert 0/24 to 12
            return f"{hours}:{minutes:02d}{period}"  # Return formatted time string
        
        # Prepare the output message
        output = f'This availability for {desired_date.date}\n'
        
        # Loop over each doctor and their available slots
        for row in rows.values:
            # row[1] is doctor_name, row[2] is the list of available slots
            output += row[1] + ". Available slots: \n" + ', \n'.join([convert_to_am_pm(value) for value in row[2]]) + '\n'

    # Return the final formatted output
    return output


In [14]:

# Example usage:
date_instance = DateModel(date="03-09-2024")
print(date_instance)

date='03-09-2024'


In [23]:
print(check_availability_by_specialization.invoke({"desired_date": date_instance, "specialization": "orthodontist"}))

This availability for 03-09-2024
kevin anderson. Available slots: 
8:00AM, 
8:30AM, 
11:30AM, 
12:00PM, 
12:30PM, 
1:00PM, 
2:00PM, 
2:30PM, 
3:00PM, 
3:30PM, 
4:00PM, 
4:30PM



In [30]:
@tool
def reschedule_appointment(old_date:DateTimeModel, new_date:DateTimeModel, id_number:IdentificationNumberModel, doctor_name:Literal['kevin anderson','robert martinez','susan davis','daniel miller','sarah wilson','michael green','lisa brown','jane smith','emily johnson','john doe']):
    """
    Rescheduling an appointment.
    The parameters MUST be mentioned by the user in the query.
    """
    #Dummy data
    df = pd.read_csv('E:\Doctor-Appointment-Aiagent\data\doctor.csv')  # You forgot to add the CSV path here
    available_for_desired_date = df[(df['date_slot'] == new_date.date)&(df['is_available'] == True)&(df['doctor_name'] == doctor_name)]
    if len(available_for_desired_date) == 0:
        return "Not available slots in the desired period"
    else:
        cancel_appointment.invoke({'date':old_date, 'id_number':id_number, 'doctor_name':doctor_name})
        set_appointment.invoke({'desired_date':new_date, 'id_number': id_number, 'doctor_name': doctor_name})
        return "Successfully rescheduled for the desired time"

In [25]:

Old_Date_Time = DateTimeModel(date="05-08-2024 08:30")
Old_Date_Time

DateTimeModel(date='05-08-2024 08:30')

In [26]:
New_Date_Time = DateTimeModel(date="28-03-2024 14:30")
New_Date_Time

DateTimeModel(date='28-03-2024 14:30')

In [27]:
IDNumber = IdentificationNumberModel(id=1000082)
IDNumber

IdentificationNumberModel(id=1000082)

In [28]:

IdentificationNumberModel(id=1000082)

IdentificationNumberModel(id=1000082)

In [ ]:
print(reschedule_appointment.invoke({"old_date": Old_Date_Time,"new_date": New_Date_Time,"id_number":IDNumber, "doctor_name": "kevin anderson"}))

Not available slots in the desired period


In [34]:
@tool
def cancel_appointment(date:DateTimeModel,id_number:IdentificationNumberModel,doctor_name:Literal['kevin anderson','robert martinez','susan davis','daniel miller','sarah wilson','michael green','lisa brown','jane smith','emily johnson','john doe']):
    """
    Canceling an appointment.
    The parameters MUST be mentioned by the user in the query.
    
    """
    df = pd.read_csv('E:\Doctor-Appointment-Aiagent\data\doctor.csv')  # You forgot to add the CSV path here
    case_to_remove = df[(df['date_slot'] == date.date)&(df['patient_to_attend'] == id_number.id)&(df['doctor_name'] == doctor_name)]
    if len(case_to_remove) == 0:
        return "You don´t have any appointment with that specifications"
    else:
        df.loc[(df['date_slot'] == date.date) & (df['patient_to_attend'] == id_number.id) & (df['doctor_name'] == doctor_name), ['is_available', 'patient_to_attend']] = [True, None]
        df.to_csv(f"../data/doctor_availability.csv", index = False)
        return "Successfully cancelled"

In [35]:

Date = DateTimeModel(date="07-08-2024 08:30")
Date

DateTimeModel(date='07-08-2024 08:30')

In [36]:
IDNumber = IdentificationNumberModel(id=1000097)
IDNumber

IdentificationNumberModel(id=1000097)

In [37]:

IdentificationNumberModel(id=1000097)

IdentificationNumberModel(id=1000097)

In [39]:

print(cancel_appointment.invoke({"date": Date,"id_number":IDNumber,"doctor_name":"john doe"}))

You don´t have any appointment with that specifications


In [40]:
@tool
def set_appointment(desired_date:DateTimeModel,id_number:IdentificationNumberModel,doctor_name:Literal['kevin anderson','robert martinez','susan davis','daniel miller','sarah wilson','michael green','lisa brown','jane smith','emily johnson','john doe']):
    """
    Set appointment or slot with the doctor.
    The parameters MUST be mentioned by the user in the query.
    """
    df = pd.read_csv('E:\Doctor-Appointment-Aiagent\data\doctor.csv')
    from datetime import datetime
    def convert_datetime_format(dt_str):
        dt = datetime.strptime(dt_str, "%d-%m-%Y %H:%M")
        return dt.strftime("%d-%m-%Y %#H.%M")
    
    case = df[(df['date_slot'] == convert_datetime_format(desired_date.date))&(df['doctor_name'] == doctor_name)&(df['is_available'] == True)]
    if len(case)==0:
         return "No available appointments for that particular case"
    else:
        df.loc[(df['date_slot'] == convert_datetime_format(desired_date.date))&(df['doctor_name'] == doctor_name) & (df['is_available'] == True), ['is_available','patient_to_attend']] = [False, id_number.id]
        df.to_csv(f"../data/doctor_availability.csv", index = False)

        return "Succesfully done"
        
        
        
    

In [41]:

Date = DateTimeModel(date="07-08-2024 08:30")
Date

DateTimeModel(date='07-08-2024 08:30')

In [42]:
IDNumber = IdentificationNumberModel(id=1000097)

In [43]:

print(set_appointment.invoke({"desired_date":Date,"id_number":IDNumber,"doctor_name":"john doe"}))

No available appointments for that particular case
